Given:

- X ∈ R^{N×D}
- y ∈ R^{N}

Implement linear regression using gradient descent:
- No loops over samples
- Add L2 regularization
- Return learned weights and loss history

Follow-ups:
- Add feature normalization
- Detect divergence / instability
- Convert to closed-form solution

# Problem Formulation
$X \in [B, D_{in}], y \in [B, D_{out}], W \in [D_{in}, D_{out}]$

$y = XW$

The loss function is l2-norm and regulariziation of weights.

$l = \frac{1}{B}||y' - y||^2_2 + \lambda ||W||^2_2$ 

The gradient is then:

$$
\frac{dl}{dW} = \frac{\partial l}{\partial y'}\frac{\partial y'}{\partial W} + \frac{\partial l}{\partial W} \\
    = 2 \frac{1}{B} (y'-y) X + 2\lambda W \\
$$

## Pytorch Implementation

In [4]:
import torch

class Model(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__() 
        self.linear = torch.nn.Linear(in_channels, out_channels)
    
    def forward(self, x):
        return self.linear(x)

# regularization term
def reg_fn(model, lamb):
    reg_loss = 0.0
    for param in model.parameters():
        reg_loss += torch.norm(param, p=2) ** 2
    return lamb * reg_loss

# total loss function
def total_loss_fn(y_pred, y, model, lamb):
    mse_loss = torch.mean((y_pred - y) ** 2)
    reg_loss = reg_fn(model, lamb)
    return mse_loss + reg_loss

# training loop
B, D_in, D_out = 100, 10, 4

# create random input and output data
X = torch.randn(B, D_in)
y = torch.randn(B, D_out)

# training settings
max_iters = 1000
lr = 1e-3
lamb = 0.01

# train
model = Model(D_in, D_out)
loss_fn = total_loss_fn
optimizer = torch.optim.SGD(model.parameters(), lr = lr)

for step in range(max_iters):
    # zero grad 
    optimizer.zero_grad()

    # forward pass 
    y_pred = model(X)

    # loss 
    loss = loss_fn(y_pred, y, model, lamb)

    # backward pass for gradients
    loss.backward() 

    # update weights using gradient descent
    optimizer.step()

    # log 
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

# print final weights
print("Final weights:", model.linear.weight.data)

Step 0, Loss: 1.3459
Step 100, Loss: 1.3049
Step 200, Loss: 1.2686
Step 300, Loss: 1.2365
Step 400, Loss: 1.2080
Step 500, Loss: 1.1827
Step 600, Loss: 1.1603
Step 700, Loss: 1.1403
Step 800, Loss: 1.1224
Step 900, Loss: 1.1065
Final weights: tensor([[ 0.0641,  0.0142,  0.0495, -0.0185,  0.0969, -0.0462,  0.0049, -0.1666,
          0.0806, -0.0532],
        [ 0.0631, -0.0685,  0.0282,  0.2614, -0.1384,  0.0297,  0.0554,  0.2042,
         -0.2239,  0.1461],
        [-0.0334,  0.0264,  0.1417,  0.1852,  0.2520,  0.0482,  0.0505, -0.1881,
         -0.0971,  0.0318],
        [ 0.1433,  0.0128,  0.0858, -0.0993,  0.0938, -0.1512,  0.0736, -0.1830,
         -0.0346, -0.1220]])


## Numpy Implementation

In [5]:
import numpy as np
import random

In [6]:
# data/model dimensions
# x: (b, d_in)
# y: (b, d_out)

B, D_in, D_out = 100, 10, 4

# create random input and output data
X = np.random.randn(B, D_in)
y = np.random.randn(B, D_out) 

# model weights 
W = np.random.randn(D_in, D_out)
#b = np.random.randn(D_out)

# training parameters
lamb = 0.01
max_iters = 1000
lr = 1e-04

In [7]:
# HELPER FUNCTIONs
def loss_fn(y_pred, y, W, lamb):
    return np.mean((y_pred - y) ** 2) + lamb * np.linalg.norm(W, 2) ** 2

def grad_fn(y_pred, y, X, W, lamb):
    grad_w = 2 * X.T @ (y_pred - y) + 2 * lamb * W
    return grad_w

In [8]:
# Linear 
for step in range(max_iters):
    y_pred = X @ W #+ b # (B, D_out)

    # loss 
    loss = loss_fn(y_pred, y, W, lamb) 

    # compute gradients
    grad_W = grad_fn(y_pred, y, X, W, lamb)

    # update weights using gradient descent 
    W = W - lr * grad_W
    #b = b - lr * grad_b

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss:.4f}")

# print final weights
print("Final weights:", W)

Step 0, Loss: 9.0031
Step 100, Loss: 1.0847
Step 200, Loss: 0.9615
Step 300, Loss: 0.9572
Step 400, Loss: 0.9569
Step 500, Loss: 0.9568
Step 600, Loss: 0.9568
Step 700, Loss: 0.9568
Step 800, Loss: 0.9568
Step 900, Loss: 0.9568
Final weights: [[ 0.00442429  0.02953795  0.22367256  0.01847641]
 [-0.00841837  0.05012672  0.14786862  0.01115062]
 [ 0.06472137 -0.14686126  0.12240796 -0.00441609]
 [-0.22802261 -0.25072843  0.11008581 -0.01429312]
 [-0.07039779  0.18259574 -0.03133131  0.13474473]
 [ 0.08284772 -0.06332044 -0.08591574  0.01918956]
 [ 0.03962742  0.03272475  0.14085466  0.0668842 ]
 [ 0.05480158 -0.11735753  0.00509636  0.03005118]
 [-0.07426909  0.08601865 -0.08259872 -0.06514633]
 [ 0.02134613  0.06032923 -0.01633676  0.08102958]]


## Analytic Solution
For the objective defined above without a bias term,

$$
L(W) = \frac{1}{B} \|XW - y\|_F^2 + \lambda \|W\|_F^2
$$

setting the gradient to zero gives

$$
\frac{2}{B} X^T (XW - y) + 2\lambda W = 0
$$

which can be rearranged into the normal equation

$$
\left(\frac{1}{B} X^T X + \lambda I\right) W = \frac{1}{B} X^T y
$$

or equivalently

$$
\left(X^T X + B\lambda I\right) W = X^T y
$$

Since $\lambda > 0$, the matrix is well-conditioned for this random example, so we can solve for $W$ directly with `np.linalg.solve`.

In [9]:
identity = np.eye(D_in)
W_analytic = np.linalg.solve(X.T @ X + B * lamb * identity, X.T @ y)
y_pred_analytic = X @ W_analytic

analytic_loss = np.sum((y_pred_analytic - y) ** 2) / B + lamb * np.sum(W_analytic ** 2)

print(f"Analytic solution loss: {analytic_loss:.4f}")
print("Analytic weights:", W_analytic)

Analytic solution loss: 3.8239
Analytic weights: [[ 0.00412048  0.02880365  0.22148491  0.01822313]
 [-0.00836881  0.05002302  0.14623805  0.01086332]
 [ 0.06406164 -0.14486843  0.12041669 -0.00435337]
 [-0.2253947  -0.24735729  0.10898858 -0.01404775]
 [-0.06997969  0.18067908 -0.03115899  0.13350416]
 [ 0.0822841  -0.06267841 -0.08497066  0.01896038]
 [ 0.03905005  0.0327388   0.13918718  0.06636721]
 [ 0.05383982 -0.11574123  0.00425371  0.02947545]
 [-0.07364185  0.08489355 -0.08181486 -0.06461548]
 [ 0.02155228  0.0597245  -0.01554777  0.08042821]]
